# Student Social Media & Wellbeing: Regression Model

Notebook này khảo sát dữ liệu, xử lý giá trị vận động âm, đánh giá mô hình trên holdout và lưu artifact dùng chung với FastAPI. Điểm `Mental_Health_Score` là điểm khảo sát, không phải chẩn đoán y tế.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "app").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from app.train import (
    DEFAULT_ARTIFACT_PATH,
    DEFAULT_METRICS_PATH,
    DEFAULT_DATA_PATH,
    load_dataset,
    prepare_dataset,
    train_model,
    save_artifacts,
)

pd.set_option("display.max_columns", None)

## 1. Đọc và khảo sát dữ liệu

In [ ]:
data = load_dataset(DEFAULT_DATA_PATH)
print(f"Kích thước: {data.shape[0]:,} dòng x {data.shape[1]} cột")
display(data.head())
display(data.dtypes.to_frame("dtype"))
display(data.isna().sum().to_frame("missing"))
display(data.describe(include="all").T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data["Mental_Health_Score"].plot(kind="hist", bins=20, ax=axes[0], title="Target score distribution")
data["Stress_Level"].value_counts().plot(kind="bar", ax=axes[1], title="Stress level counts")
axes[0].set_xlabel("Mental health score")
axes[1].set_ylabel("Rows")
plt.tight_layout()
plt.show()

negative_activity = int((data["Physical_Activity_Hours"] < 0).sum())
print(f"Số giá trị Physical_Activity_Hours âm cần chặn về 0: {negative_activity}")

## 2. Tiền xử lý và đánh giá

Mô hình dùng One-Hot Encoding cho dữ liệu phân loại và Random Forest cho hồi quy. Tập kiểm tra 20% được giữ riêng để báo cáo MAE, RMSE và R²; sau đánh giá, pipeline được fit lại bằng toàn bộ dữ liệu để phục vụ API.

In [ ]:
X, y, cleaning_report = prepare_dataset(data)
print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Cleaning report:", cleaning_report)

model, metrics, evaluation, cleaning_report = train_model(data)
print("Holdout metrics:")
for name, value in metrics.items():
    print(f"{name.upper()}: {value:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(evaluation["actual"], evaluation["predicted"], alpha=0.45)
limits = [min(evaluation.min()), max(evaluation.max())]
ax.plot(limits, limits, "--", color="red", label="Perfect prediction")
ax.set(xlabel="Actual score", ylabel="Predicted score", title="Holdout: actual vs predicted")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Lưu model cho API

Chạy cell này để tạo/cập nhật `artifacts/mental_health_model.joblib` và `artifacts/metrics.json`.

In [ ]:
metadata = save_artifacts(
    model=model,
    metrics=metrics,
    training_rows=len(data),
    cleaning_report=cleaning_report,
    artifact_path=DEFAULT_ARTIFACT_PATH,
    metrics_path=DEFAULT_METRICS_PATH,
)
metadata